# 2022 국정감사 처리결과 Highlight 기반 요구사항 매핑

이 노트북은 2022년도 PDF Highlight annotation을 최종 텍스트가 아닌 **표 행 선택 프록시**로 사용한다. annotation → 목차/본문 행 → `issue_no` → 좌우 셀 전체 텍스트 → anchor 분리 → 상태 분류 → 조건부 OCR → 2차 전수검토 → 연도별 export를 한 번에 실행한다.

2021~2023은 이 실행 범위에서 제외하며, 다른 노트북의 함수·상태를 import하지 않는다.

## 00. 실행 계약·환경·경로

상수, 플래그, schema, 로깅과 출력 경로를 고정한다. 별도 Python 모듈이나 실행 스크립트는 사용하지 않는다.

In [1]:
from __future__ import annotations

import sys
import site
from pathlib import Path

# dsja_sbs_venv에서도 사용자 설치 PyMuPDF를 명시적으로 찾는다.
_user_site = site.getusersitepackages()
if _user_site not in sys.path:
    sys.path.append(_user_site)

import csv
import hashlib
import json
import logging
import re
import shutil
import subprocess
import tempfile
import textwrap
from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher
from typing import Any

import fitz
import pandas as pd

PROJECT_ROOT = Path("/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET")
NOTEBOOK_PATH = PROJECT_ROOT / "03_2022_marked_audit_result.ipynb"
INPUT_PDF_DIR = PROJECT_ROOT / "inputs" / "pdf"
INPUT_REGISTRY_DIR = PROJECT_ROOT / "inputs" / "registry"
LEGACY_MARKED_PDF_DIR = PROJECT_ROOT / "target_mark_pdf"
OUT_ROOT = PROJECT_ROOT / "outputs" / "marked_parallel" / "2022"
LOG_DIR = OUT_ROOT / "logs"
PARQUET_PATH = OUT_ROOT / "2022_marked_issue_mapping.parquet"
REVIEW_CSV_PATH = OUT_ROOT / "2022_marked_issue_mapping.csv"
MANIFEST_PATH = OUT_ROOT / "2022_pipeline_manifest.json"
RUN_REPORT_PATH = OUT_ROOT / "2022_RUN_REPORT.md"
QUALITY_REPORT_PATH = PROJECT_ROOT / "reports" / "2022_MARKED_MAPPING_QUALITY.md"
LOG_PATH = LOG_DIR / "2022_marked_audit_result.log"

for directory in [INPUT_PDF_DIR, INPUT_REGISTRY_DIR, OUT_ROOT, LOG_DIR, QUALITY_REPORT_PATH.parent]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_PDF_AUDIT = True
RUN_ANNOTATION_PARSE = True
RUN_TOC_PARSE = True
RUN_BODY_PARSE = True
RUN_MAPPING = True
RUN_NATIVE_STATUS = True
RUN_OCR_FALLBACK = True
RUN_EXPORT = True

RUN_FULL_PAGE_OCR = False
RUN_TFIDF = False
RUN_EMBEDDING = False
RUN_SQLITE = False
RUN_LLM = False

REQUESTED_REPORT_YEAR = "2022"
CONTROLLED_PLACEHOLDER_MEETING_ID = "UNMAPPED_REPORT_2022"
CANONICAL_COLUMNS = [
    "issue_id", "meeting_id", "issue_no", "issue_text", "action_text",
    "future_plan_text", "status", "source_page", "parse_source",
]
REVIEW_COLUMNS = [
    "meeting_id", "meeting_year", "issue_no", "issue_text",
    "action_text", "future_plan_text", "status", "source_page",
]
ALLOWED_STATUS = ["complete", "active", "uncomplete", "null"]
ALLOWED_PARSE_SOURCE = ["native", "ocr", "native_with_ocr_status"]

ISSUE_NO_PATTERN = re.compile(r"^\s*(\d{1,3})\s*[.]")
ISSUE_TOKEN_PATTERN = re.compile(r"^(\d{1,3})\.?$")
ACTION_ANCHOR_PATTERN = re.compile(r"<\s*(?:조치\s*사항|조치\s*중)\s*>")
FUTURE_ANCHOR_PATTERN = re.compile(r"<\s*향후\s*추진\s*계획\s*>")
COMBINED_ANCHOR_PATTERN = re.compile(r"<\s*조치\s*(?:사항\s*)?(?:및|·|/)?\s*향후\s*(?:추진\s*)?계획\s*>")

logging.getLogger("marked_audit").handlers.clear()
logger = logging.getLogger("marked_audit")
logger.setLevel(logging.INFO)
_file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
_file_handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(_file_handler)

RUN_STARTED_AT = datetime.now().astimezone()
warnings: list[str] = []
logger.info("pipeline_start project_root=%s", PROJECT_ROOT)

print({
    "project_root": str(PROJECT_ROOT),
    "notebook": str(NOTEBOOK_PATH),
    "flags_enabled": [k for k, v in list(globals().items()) if k.startswith("RUN_") and v],
})

{'project_root': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET', 'notebook': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/03_2022_marked_audit_result.ipynb', 'flags_enabled': ['RUN_REPORT_PATH', 'RUN_PDF_AUDIT', 'RUN_ANNOTATION_PARSE', 'RUN_TOC_PARSE', 'RUN_BODY_PARSE', 'RUN_MAPPING', 'RUN_NATIVE_STATUS', 'RUN_OCR_FALLBACK', 'RUN_EXPORT', 'RUN_STARTED_AT']}


## 01. 2022 입력 PDF 및 Registry 확인

`inputs/pdf`를 우선 확인하고 실제 공급 위치인 `target_mark_pdf`를 controlled fallback으로 사용한다. title page에서 2022이 확인된 PDF만 후보로 남기며, 같은 연도의 복수본은 annotation이 가장 충실한 통합본을 선택한다.

In [2]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def count_highlights(doc: fitz.Document) -> int:
    total = 0
    for page in doc:
        annot = page.first_annot
        while annot:
            total += int(annot.type[1] == "Highlight")
            annot = annot.next
    return total


def inspect_pdf(path: Path) -> dict[str, Any]:
    doc = fitz.open(path)
    first_page_text = doc[0].get_text("text")
    year_match = re.search(r"(20\d{2})\s*년도", first_page_text)
    info = {
        "path": str(path),
        "filename": path.name,
        "sha256": sha256_file(path),
        "page_count": len(doc),
        "highlight_count": count_highlights(doc),
        "detected_report_year": year_match.group(1) if year_match else None,
        "title_page_excerpt": re.sub(r"\s+", " ", first_page_text).strip()[:250],
    }
    doc.close()
    return info


preferred_names = [
    f"{REQUESTED_REPORT_YEAR}년도 국정감사결과 시정 및 처리 요구사항에 대한 처리결과 보고서 (1).pdf",
    f"{REQUESTED_REPORT_YEAR}년도 국정감사결과 시정 및 처리 요구사항에 대한 처리결과 보고서.pdf",
]

explicit_input_pdf = LEGACY_MARKED_PDF_DIR / "2022_국정감사결과_요구사항1-259_260-끝_병합.pdf"
pdf_source_dir = LEGACY_MARKED_PDF_DIR
all_pdf_candidates = [explicit_input_pdf] if explicit_input_pdf.exists() else []

if not all_pdf_candidates:
    raise FileNotFoundError("No input PDF found in inputs/pdf or target_mark_pdf")

all_pdf_audits = [inspect_pdf(p) for p in all_pdf_candidates] if RUN_PDF_AUDIT else []
pdf_audits = [
    x for x in all_pdf_audits
    if str(x.get("detected_report_year")) == REQUESTED_REPORT_YEAR
]
pdf_candidates = [Path(x["path"]) for x in pdf_audits]
audit_by_name = {x["filename"]: x for x in pdf_audits}

if not pdf_candidates:
    raise FileNotFoundError(f"No {REQUESTED_REPORT_YEAR} PDF found after title-page year validation")

exact = [p for p in pdf_candidates if p.name in preferred_names]
if len(exact) == 1:
    input_pdf = exact[0]
    pdf_selection_rule = "exact_year_candidate_name"
elif len(pdf_candidates) == 1:
    input_pdf = pdf_candidates[0]
    pdf_selection_rule = "single_year_validated_pdf"
else:
    integrated = [p for p in pdf_candidates if "형광펜" in p.name and "통합본" in p.name]
    if len(integrated) != 1:
        raise RuntimeError(f"Ambiguous {REQUESTED_REPORT_YEAR} input PDFs: {[p.name for p in pdf_candidates]}")
    input_pdf = integrated[0]
    other_audits = [x for x in pdf_audits if x["filename"] != input_pdf.name]
    selected_audit = audit_by_name[input_pdf.name]
    if other_audits and (
        any(x["page_count"] != selected_audit["page_count"] for x in other_audits)
        or any(x["highlight_count"] > selected_audit["highlight_count"] for x in other_audits)
    ):
        raise RuntimeError("Integrated PDF is not the page-compatible highlight-rich candidate")
    pdf_selection_rule = "marked_integrated_highlight_rich_version"
    warnings.append("Multiple same-year PDFs audited sequentially; marked integrated version selected")

input_audit = audit_by_name[input_pdf.name]
PDF_SHA256 = input_audit["sha256"]
PAGE_COUNT = input_audit["page_count"]
DETECTED_REPORT_YEAR = input_audit["detected_report_year"]
if DETECTED_REPORT_YEAR != REQUESTED_REPORT_YEAR:
    warnings.append(
        f"Requested report year {REQUESTED_REPORT_YEAR}, but title page detected {DETECTED_REPORT_YEAR}"
    )

COLUMN_ALIAS_MAP = {
    "회의ID": "meeting_id", "회의 ID": "meeting_id",
    "대수": "meeting_number", "국회대수": "meeting_number",
    "회기": "meeting_count",
    "회의일자": "meeting_date_raw", "회의 일자": "meeting_date_raw",
    "다운로드 URL": "source_url", "다운로드URL": "source_url",
    "PDF URL": "source_url", "URL": "source_url",
    "위원회코드": "committee_code", "위원회명": "committee_name",
}
REGISTRY_CANONICAL_COLUMNS = [
    "meeting_id", "meeting_number", "meeting_count", "meeting_year",
    "meeting_date", "source_url", "committee_code", "committee_name",
]
allowed_registry_candidates = [
    PROJECT_ROOT.parent / "data_parse" / "pdf_crawler" / "control_registry.parquet",
    PROJECT_ROOT.parent / "P3_PROJECT" / "inputs" / "registry" / "국정감사회의록_문화체육 2020~.xlsx",
]
local_registry_candidates = sorted(INPUT_REGISTRY_DIR.glob("*.parquet")) + sorted(INPUT_REGISTRY_DIR.glob("*.xlsx"))
registry_candidates = local_registry_candidates + [p for p in allowed_registry_candidates if p.exists()]


def normalize_registry(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.rename(columns={c: COLUMN_ALIAS_MAP.get(str(c).strip(), str(c).strip()) for c in raw.columns}).copy()
    if "meeting_id" in df:
        df["meeting_id"] = df["meeting_id"].astype("string")
    if "meeting_number" in df:
        df["meeting_number"] = df["meeting_number"].astype("string").str.extract(r"(\d+)")[0]
    if "meeting_count" in df:
        df["meeting_count"] = df["meeting_count"].astype("string").str.extract(r"(\d+)")[0]
    date_col = "meeting_date_raw" if "meeting_date_raw" in df else ("meeting_date" if "meeting_date" in df else None)
    if date_col:
        dt = pd.to_datetime(df[date_col], errors="coerce")
        df["meeting_year"] = dt.dt.strftime("%y")
        df["meeting_date"] = dt.dt.strftime("%m%d")
    return df


def load_registry(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return normalize_registry(pd.read_parquet(path))
    return normalize_registry(pd.read_excel(path, dtype=str))


def registry_match(df: pd.DataFrame, pdf_path: Path, sha256: str, report_year: str | None) -> pd.DataFrame:
    if df.empty:
        return df
    mask = pd.Series(False, index=df.index)
    if "source_url" in df:
        source_name = df["source_url"].astype("string").str.replace("\\", "/", regex=False).str.rsplit("/", n=1).str[-1]
        mask |= source_name.fillna("").eq(pdf_path.name)
    for col in ["filename", "pdf_filename", "document_title", "title"]:
        if col in df:
            mask |= df[col].astype("string").fillna("").str.contains(
                re.escape(pdf_path.stem[:30]), regex=True
            )
    for col in ["sha256", "pdf_sha256"]:
        if col in df:
            mask |= df[col].astype("string").str.lower().eq(sha256.lower())
    candidates = df.loc[mask].copy()
    return candidates


registry_path: Path | None = None
registry_df = pd.DataFrame()
registry_candidates_matched = pd.DataFrame()
if registry_candidates:
    parquet_first = sorted(registry_candidates, key=lambda p: (p.suffix.lower() != ".parquet", str(p)))
    registry_path = parquet_first[0]
    registry_df = load_registry(registry_path)
    registry_candidates_matched = registry_match(
        registry_df, input_pdf, PDF_SHA256, DETECTED_REPORT_YEAR
    )

if len(registry_candidates_matched) == 1:
    registry_row = registry_candidates_matched.iloc[0]
    meeting_id = str(registry_row["meeting_id"])
    meeting_year = str(registry_row.get("meeting_year", "")) or str(DETECTED_REPORT_YEAR)[-2:]
    registry_mapping_status = "mapped"
    registry_mapping_warning = None
else:
    meeting_id = CONTROLLED_PLACEHOLDER_MEETING_ID
    meeting_year = REQUESTED_REPORT_YEAR[-2:]
    registry_mapping_status = "unmapped"
    registry_mapping_warning = "No unique meeting_id matched"
    warnings.append(registry_mapping_warning)

logger.info(
    "input_selected file=%s sha256=%s pages=%s selection=%s registry=%s meeting_id=%s",
    input_pdf.name, PDF_SHA256, PAGE_COUNT, pdf_selection_rule,
    registry_mapping_status, meeting_id,
)
print({
    "pdf_audits": [
        {"filename": x["filename"], "pages": x["page_count"], "highlights": x["highlight_count"]}
        for x in pdf_audits
    ],
    "selected_pdf": str(input_pdf),
    "sha256": PDF_SHA256,
    "page_count": PAGE_COUNT,
    "detected_report_year": DETECTED_REPORT_YEAR,
    "registry": str(registry_path) if registry_path else None,
    "meeting_id": meeting_id,
    "registry_mapping": registry_mapping_status,
})

{'pdf_audits': [{'filename': '2022_국정감사결과_요구사항1-259_260-끝_병합.pdf', 'pages': 159, 'highlights': 246}], 'selected_pdf': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/target_mark_pdf/2022_국정감사결과_요구사항1-259_260-끝_병합.pdf', 'sha256': '9390284efe2f80a8b544cfe2da900805ed02941db8a1a7a37a5dec960fb0b4cb', 'page_count': 159, 'detected_report_year': '2022', 'registry': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/data_parse/pdf_crawler/control_registry.parquet', 'meeting_id': 'UNMAPPED_REPORT_2022', 'registry_mapping': 'unmapped'}


## 02. PDF Annotation Audit

PyMuPDF로 Highlight의 물리 페이지, xref, rect, quad points, 색상, 덮는 native text, 목차/본문 구역과 좌·우 셀 위치를 기록한다. notebook output에는 본문 전체를 출력하지 않는다.

In [3]:
def page_vertical_divider(page: fitz.Page) -> float:
    candidates: list[float] = []
    width = page.rect.width
    for drawing in page.get_drawings():
        for item in drawing.get("items", []):
            if item[0] != "l":
                continue
            a, b = item[1], item[2]
            if (
                abs(a.x - b.x) < 0.8
                and abs(a.y - b.y) > 20
                and width * 0.25 < a.x < width * 0.55
            ):
                candidates.append(round(float(a.x), 1))
    if candidates:
        return float(Counter(candidates).most_common(1)[0][0])
    return float(width * 0.39)


def body_issue_starts(page: fitz.Page) -> list[dict[str, float | int]]:
    starts = []
    for word in page.get_text("words"):
        x0, y0, x1, y1, token, *_ = word
        m = ISSUE_TOKEN_PATTERN.fullmatch(token.strip())
        if x0 < 62 and y0 > 85 and m:
            starts.append({"issue_no": int(m.group(1)), "y0": float(y0), "y1": float(y1)})
    return sorted(starts, key=lambda x: x["y0"])


def detect_body_start(doc: fitz.Document) -> int:
    for page_index, page in enumerate(doc):
        text = page.get_text("text")
        starts = body_issue_starts(page)
        if "시정및처리요구사항" in text and any(x["issue_no"] == 1 for x in starts):
            return page_index + 1
    raise RuntimeError("Body start page was not detected")


def annotation_side(rect: fitz.Rect, page: fitz.Page, body_start_page: int, physical_page: int) -> str:
    if physical_page == 1:
        return "front_matter"
    if physical_page < body_start_page:
        return "toc"
    divider = page_vertical_divider(page)
    if rect.x1 <= divider + 2:
        return "left"
    if rect.x0 >= divider - 2:
        return "right"
    return "both"


def parse_annotations(path: Path) -> tuple[pd.DataFrame, int]:
    doc = fitz.open(path)
    body_start_page = detect_body_start(doc)
    rows: list[dict[str, Any]] = []
    for page_index, page in enumerate(doc):
        annot = page.first_annot
        while annot:
            if annot.type[1] == "Highlight":
                vertices = []
                if annot.vertices:
                    vertices = [
                        [round(float(p.x), 3), round(float(p.y), 3)]
                        if hasattr(p, "x") else [round(float(p[0]), 3), round(float(p[1]), 3)]
                        for p in annot.vertices
                    ]
                rect = annot.rect
                colors = annot.colors or {}
                rows.append({
                    "physical_page": page_index + 1,
                    "annotation_type": annot.type[1],
                    "xref": int(annot.xref),
                    "rect_x0": float(rect.x0), "rect_y0": float(rect.y0),
                    "rect_x1": float(rect.x1), "rect_y1": float(rect.y1),
                    "quad_points": vertices,
                    "colors": colors,
                    "covered_native_text": re.sub(r"\s+", " ", page.get_textbox(rect)).strip(),
                    "region": (
                        "front_matter" if page_index == 0
                        else ("toc" if page_index + 1 < body_start_page else "body")
                    ),
                    "cell_side": annotation_side(rect, page, body_start_page, page_index + 1),
                })
            annot = annot.next
    doc.close()
    return pd.DataFrame(rows), body_start_page


if not RUN_ANNOTATION_PARSE:
    raise RuntimeError("Annotation parse is required for this execution contract")

annotation_df, BODY_START_PAGE = parse_annotations(input_pdf)
if annotation_df.empty:
    raise RuntimeError("Highlight annotation count is zero")

front_matter_annotation_count = int(annotation_df["region"].eq("front_matter").sum())
annotation_side_counts = annotation_df["cell_side"].value_counts().to_dict()
logger.info("annotation_audit highlights=%d body_start_page=%d", len(annotation_df), BODY_START_PAGE)
print({
    "body_start_page": BODY_START_PAGE,
    "highlight_annotations": len(annotation_df),
    "region_counts": annotation_df["region"].value_counts().to_dict(),
    "cell_side_counts": annotation_side_counts,
})

{'body_start_page': 26, 'highlight_annotations': 246, 'region_counts': {'body': 144, 'toc': 101, 'front_matter': 1}, 'cell_side_counts': {'both': 103, 'toc': 101, 'right': 22, 'left': 19, 'front_matter': 1}}


## 03. 목차 Marker Parser

목차의 일련번호 token과 다음 일련번호 시작점을 이용해 entry geometry를 구성한다. 페이지 상단의 연속 fragment도 직전 issue에 귀속하고 marker 중심점과 행 구간을 교차시킨다.

In [4]:
def normalize_text(value: Any) -> str:
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def sorted_words_in_band(
    page: fitz.Page, top: float, bottom: float, x_min: float = 0, x_max: float | None = None
) -> list[tuple]:
    x_max = float(page.rect.width) if x_max is None else x_max
    selected = []
    for word in page.get_text("words"):
        x0, y0, x1, y1, token, *_ = word
        cy = (y0 + y1) / 2
        if top <= cy < bottom and x_min <= (x0 + x1) / 2 < x_max:
            selected.append(word)
    return sorted(selected, key=lambda w: (round(float(w[1]), 1), float(w[0])))


def words_to_text(words: list[tuple]) -> str:
    return normalize_text(" ".join(str(w[4]) for w in words))


def toc_issue_starts(page: fitz.Page) -> list[dict[str, float | int]]:
    starts = []
    for word in page.get_text("words"):
        x0, y0, x1, y1, token, *_ = word
        m = ISSUE_TOKEN_PATTERN.fullmatch(token.strip())
        if x0 < 100 and token.strip().endswith(".") and m:
            starts.append({"issue_no": int(m.group(1)), "y0": float(y0), "y1": float(y1)})
    return sorted(starts, key=lambda x: x["y0"])


def parse_toc(doc: fitz.Document, body_start_page: int) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    records: dict[int, dict[str, Any]] = {}
    segments: list[dict[str, Any]] = []
    previous_issue: int | None = None

    for page_index in range(1, body_start_page - 1):
        page = doc[page_index]
        starts = toc_issue_starts(page)
        if starts and previous_issue is not None and starts[0]["y0"] > 50:
            segment = {
                "physical_page": page_index + 1, "top": 50.0,
                "bottom": float(starts[0]["y0"]), "issue_no": previous_issue,
                "segment_type": "continuation",
            }
            segments.append(segment)
            continuation = words_to_text(sorted_words_in_band(page, segment["top"], segment["bottom"], 45))
            if continuation and previous_issue in records:
                records[previous_issue]["toc_title_raw"] += " " + continuation

        for j, start in enumerate(starts):
            issue_no = int(start["issue_no"])
            bottom = float(starts[j + 1]["y0"]) if j + 1 < len(starts) else 780.0
            segment = {
                "physical_page": page_index + 1, "top": float(start["y0"]),
                "bottom": bottom, "issue_no": issue_no, "segment_type": "start",
            }
            segments.append(segment)
            entry = words_to_text(sorted_words_in_band(page, segment["top"], segment["bottom"], 45))
            entry = re.sub(rf"^\s*{issue_no}\s*[.]\s*", "", entry)
            printed = re.search(r"(\d{1,3})\s*$", entry)
            printed_page = int(printed.group(1)) if printed else pd.NA
            title = entry[: printed.start()].strip() if printed else entry
            title = re.sub(r"[·.]{3,}", " ", title)
            records[issue_no] = {
                "issue_no": issue_no,
                "toc_title_raw": normalize_text(title),
                "printed_page": printed_page,
                "toc_page": page_index + 1,
            }
            previous_issue = issue_no

    toc_df = pd.DataFrame(records.values()).sort_values("issue_no").reset_index(drop=True)
    toc_df["issue_no"] = toc_df["issue_no"].astype("Int64")
    toc_df["printed_page"] = toc_df["printed_page"].astype("Int64")
    return toc_df, segments


def map_annotation_rows(
    annotations: pd.DataFrame, segments: list[dict[str, Any]], region: str
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    by_page: dict[int, list[dict[str, Any]]] = defaultdict(list)
    for segment in segments:
        by_page[int(segment["physical_page"])].append(segment)
    mapped, unresolved = [], []
    subset = annotations[annotations["region"].eq(region)]
    for row in subset.to_dict("records"):
        page_segments = by_page.get(int(row["physical_page"]), [])
        rect_top = float(row["rect_y0"])
        rect_bottom = float(row["rect_y1"])
        cy = (rect_top + rect_bottom) / 2
        candidates = [
            s for s in page_segments
            if min(rect_bottom, float(s["bottom"]) + 3)
            - max(rect_top, float(s["top"]) - 3) > 0
        ]
        if not candidates:
            unresolved.append(row)
            continue
        chosen = max(
            candidates,
            key=lambda s: (
                min(rect_bottom, float(s["bottom"]) + 3)
                - max(rect_top, float(s["top"]) - 3),
                -abs((float(s["top"]) + float(s["bottom"])) / 2 - cy),
            ),
        )
        mapped.append({**row, **{f"segment_{k}": v for k, v in chosen.items()}})
    return mapped, unresolved


doc = fitz.open(input_pdf)
toc_df, toc_segments = parse_toc(doc, BODY_START_PAGE)
toc_annotation_map, unresolved_toc_annotations = map_annotation_rows(
    annotation_df, toc_segments, "toc"
)
toc_marked_issue_nos = {int(x["segment_issue_no"]) for x in toc_annotation_map}
toc_df["marker_present"] = toc_df["issue_no"].isin(toc_marked_issue_nos)
toc_df["marker_count"] = toc_df["issue_no"].map(
    Counter(int(x["segment_issue_no"]) for x in toc_annotation_map)
).fillna(0).astype(int)

logger.info(
    "toc_parse entries=%d marked_issues=%d mapped_annotations=%d unresolved=%d",
    len(toc_df), len(toc_marked_issue_nos), len(toc_annotation_map), len(unresolved_toc_annotations),
)
print({
    "toc_entries": len(toc_df),
    "toc_marked_issues": len(toc_marked_issue_nos),
    "toc_annotations_mapped": len(toc_annotation_map),
    "toc_annotations_unresolved": len(unresolved_toc_annotations),
})

{'toc_entries': 502, 'toc_marked_issues': 96, 'toc_annotations_mapped': 101, 'toc_annotations_unresolved': 0}


## 04. 본문 표 Geometry 및 행 Parser

반복 vertical vector x좌표로 페이지별 divider를 찾고, 실패 시 page width 비율을 사용한다. 왼쪽의 번호 token과 다음 번호 시작점을 행 경계로 삼아 좌우 cell word를 분리하고, 페이지 상단 continuation을 직전 issue fragment에 병합한다.

In [5]:
def table_content_top(page: fitz.Page) -> float:
    header_bottoms = [
        float(w[3]) for w in page.get_text("words")
        if str(w[4]) in {"시정및처리요구사항", "조치사항및향후추진계획"}
    ]
    return max(header_bottoms) + 1.0 if header_bottoms else 95.0


def grab_cells(page: fitz.Page, top: float, bottom: float, divider: float) -> tuple[str, str]:
    left_words, right_words = [], []
    for word in page.get_text("words"):
        x0, y0, x1, y1, token, *_ = word
        cy = (y0 + y1) / 2
        if not (top <= cy < bottom):
            continue
        if (x0 + x1) / 2 < divider:
            left_words.append(word)
        else:
            right_words.append(word)
    return words_to_text(sorted(left_words, key=lambda w: (round(float(w[1]), 1), float(w[0])))), words_to_text(
        sorted(right_words, key=lambda w: (round(float(w[1]), 1), float(w[0])))
    )


def append_fragment(record: dict[str, Any], page_no: int, top: float, bottom: float, divider: float,
                    left: str, right: str, segment_type: str) -> None:
    if left:
        record["issue_text_raw"] = normalize_text(record["issue_text_raw"] + " " + left)
    if right:
        record["response_raw"] = normalize_text(record["response_raw"] + " " + right)
    record["fragments"].append({
        "physical_page": page_no, "top": float(top), "bottom": float(bottom),
        "divider": float(divider), "segment_type": segment_type,
    })


def parse_body(doc: fitz.Document, body_start_page: int) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    records: list[dict[str, Any]] = []
    segments: list[dict[str, Any]] = []
    current: dict[str, Any] | None = None
    occurrence = 0

    for page_index in range(body_start_page - 1, len(doc)):
        page = doc[page_index]
        page_no = page_index + 1
        divider = page_vertical_divider(page)
        starts = body_issue_starts(page)
        top = table_content_top(page)

        if current is not None and starts and float(starts[0]["y0"]) > top:
            upper = float(starts[0]["y0"])
            left, right = grab_cells(page, top, upper, divider)
            is_label_only = bool(left) and not right and (
                left.startswith("(") or left.startswith("【")
            )
            if (left or right) and not is_label_only:
                append_fragment(current, page_no, top, upper, divider, left, right, "continuation")
                segments.append({
                    "physical_page": page_no, "top": top, "bottom": upper,
                    "issue_no": current["issue_no"], "occurrence": current["occurrence"],
                    "segment_type": "continuation", "divider": divider,
                })
        elif current is not None and not starts:
            left, right = grab_cells(page, top, 775.0, divider)
            if left or right:
                append_fragment(current, page_no, top, 775.0, divider, left, right, "continuation")
                segments.append({
                    "physical_page": page_no, "top": top, "bottom": 775.0,
                    "issue_no": current["issue_no"], "occurrence": current["occurrence"],
                    "segment_type": "continuation", "divider": divider,
                })

        for j, start in enumerate(starts):
            issue_no = int(start["issue_no"])
            row_top = float(start["y0"])
            row_bottom = float(starts[j + 1]["y0"]) if j + 1 < len(starts) else 775.0
            left, right = grab_cells(page, row_top, row_bottom, divider)
            occurrence += 1
            current = {
                "occurrence": occurrence,
                "issue_no": issue_no,
                "source_page": page_no,
                "issue_text_raw": "",
                "response_raw": "",
                "fragments": [],
            }
            append_fragment(current, page_no, row_top, row_bottom, divider, left, right, "start")
            records.append(current)
            segments.append({
                "physical_page": page_no, "top": row_top, "bottom": row_bottom,
                "issue_no": issue_no, "occurrence": occurrence,
                "segment_type": "start", "divider": divider,
            })

    return records, segments


body_records, body_segments = parse_body(doc, BODY_START_PAGE)
body_annotation_map, unresolved_body_annotations = map_annotation_rows(
    annotation_df, body_segments, "body"
)
body_marker_counts = Counter(int(x["segment_issue_no"]) for x in body_annotation_map)
body_marked_occurrences = {int(x["segment_occurrence"]) for x in body_annotation_map}
for record in body_records:
    record["marker_present"] = int(record["occurrence"]) in body_marked_occurrences
    record["marker_count"] = sum(
        1 for x in body_annotation_map if int(x["segment_occurrence"]) == int(record["occurrence"])
    )

body_issue_counts = Counter(int(r["issue_no"]) for r in body_records)
body_numbering_defects = {
    "missing_issue_nos": sorted(set(range(1, int(toc_df["issue_no"].max()) + 1)) - set(body_issue_counts)),
    "duplicate_issue_nos": sorted([n for n, c in body_issue_counts.items() if c > 1]),
}
if body_numbering_defects["missing_issue_nos"] or body_numbering_defects["duplicate_issue_nos"]:
    warnings.append(f"Native body numbering defect: {body_numbering_defects}")

logger.info(
    "body_parse rows=%d unique_issue_nos=%d marked_occurrences=%d mapped_annotations=%d unresolved=%d",
    len(body_records), len(body_issue_counts), len(body_marked_occurrences),
    len(body_annotation_map), len(unresolved_body_annotations),
)
print({
    "body_rows": len(body_records),
    "body_unique_issue_nos": len(body_issue_counts),
    "body_marked_occurrences": len(body_marked_occurrences),
    "body_annotations_mapped": len(body_annotation_map),
    "body_annotations_unresolved": len(unresolved_body_annotations),
    "body_numbering_defects": body_numbering_defects,
})

{'body_rows': 503, 'body_unique_issue_nos': 503, 'body_marked_occurrences': 113, 'body_annotations_mapped': 144, 'body_annotations_unresolved': 0, 'body_numbering_defects': {'missing_issue_nos': [], 'duplicate_issue_nos': []}}


## 05. Marker-to-Issue Mapping

목차 또는 본문 중 하나라도 마킹된 번호를 보존한다. exact `issue_no`를 우선하고, 번호 불일치 시 임의로 다른 번호를 선택하지 않는다. title similarity는 후보 감사값으로만 남기며 자동 재번호화에는 사용하지 않는다.

In [6]:
def normalized_similarity_text(value: str) -> str:
    return re.sub(r"[^0-9A-Za-z가-힣]", "", normalize_text(value)).lower()


body_by_issue: dict[int, list[dict[str, Any]]] = defaultdict(list)
for record in body_records:
    body_by_issue[int(record["issue_no"])].append(record)

body_marked_issue_nos = {
    int(record["issue_no"]) for record in body_records if record["marker_present"]
}
selected_issue_nos = sorted(toc_marked_issue_nos | body_marked_issue_nos)

mapping_rows: list[dict[str, Any]] = []
selected_records: list[dict[str, Any]] = []
for issue_no in selected_issue_nos:
    candidates = body_by_issue.get(issue_no, [])
    toc_row = toc_df.loc[toc_df["issue_no"].eq(issue_no)]
    toc_title = "" if toc_row.empty else str(toc_row.iloc[0]["toc_title_raw"])
    marker_source = (
        "toc_and_body" if issue_no in toc_marked_issue_nos and issue_no in body_marked_issue_nos
        else "toc_only" if issue_no in toc_marked_issue_nos else "body_only"
    )
    if len(candidates) == 1:
        chosen = candidates[0]
        mapping_method = "issue_no_exact"
        mapping_status = "mapped"
        title_similarity = SequenceMatcher(
            None,
            normalized_similarity_text(toc_title),
            normalized_similarity_text(chosen["issue_text_raw"]),
        ).ratio() if toc_title else None
        selected_records.append(chosen)
    else:
        chosen = None
        mapping_method = "unresolved"
        mapping_status = "unresolved"
        title_similarity = None
    mapping_rows.append({
        "issue_no": issue_no,
        "marker_source": marker_source,
        "body_candidate_count": len(candidates),
        "mapping_method": mapping_method,
        "mapping_status": mapping_status,
        "title_similarity_audit": title_similarity,
        "source_page": chosen["source_page"] if chosen else pd.NA,
        "occurrence": chosen["occurrence"] if chosen else pd.NA,
    })

mapping_df = pd.DataFrame(mapping_rows)
mapped_issue_count = int(mapping_df["mapping_status"].eq("mapped").sum())
unresolved_issue_count = int(mapping_df["mapping_status"].eq("unresolved").sum())
marker_source_counts = mapping_df["marker_source"].value_counts().to_dict()

logger.info(
    "mapping selected=%d mapped=%d unresolved=%d source_counts=%s",
    len(selected_issue_nos), mapped_issue_count, unresolved_issue_count, marker_source_counts,
)
print({
    "selected_issues": len(selected_issue_nos),
    "mapping_success": mapped_issue_count,
    "mapping_unresolved": unresolved_issue_count,
    "marker_source_counts": marker_source_counts,
})

{'selected_issues': 116, 'mapping_success': 116, 'mapping_unresolved': 0, 'marker_source_counts': {'toc_and_body': 93, 'body_only': 20, 'toc_only': 3}}


## 06. Issue Text Assembly

선택된 본문 행의 왼쪽 cell fragment를 순서대로 결합한다. issue 번호 prefix는 별도 key에 이미 보존되므로 제거하고, 의미 단어는 삭제하지 않은 채 공백만 정규화한다.

In [7]:
def clean_issue_text(raw: str, issue_no: int) -> str:
    text = normalize_text(raw)
    text = re.sub(rf"^\s*{issue_no}\s*[.]?\s*", "", text, count=1)
    text = re.sub(r"\s+-\s+\d+\s+-\s*$", "", text)
    return normalize_text(text)


def clean_response_text(raw: str) -> str:
    text = normalize_text(raw)
    text = text.replace("〈", "<").replace("〉", ">")
    text = re.sub(r"\s+-\s+\d+\s+-\s*$", "", text)
    return normalize_text(text)


assembled_rows: list[dict[str, Any]] = []
selected_occurrences = {int(x["occurrence"]) for x in selected_records}
for record in selected_records:
    assembled_rows.append({
        "occurrence": int(record["occurrence"]),
        "issue_no": int(record["issue_no"]),
        "issue_text": clean_issue_text(record["issue_text_raw"], int(record["issue_no"])),
        "response_native": clean_response_text(record["response_raw"]),
        "source_page": int(record["source_page"]),
        "fragments": record["fragments"],
    })

issue_assembly_df = pd.DataFrame(assembled_rows).sort_values("issue_no").reset_index(drop=True)
empty_issue_count = int(issue_assembly_df["issue_text"].eq("").sum())
empty_response_count = int(issue_assembly_df["response_native"].eq("").sum())

print({
    "assembled_issues": len(issue_assembly_df),
    "empty_issue_text": empty_issue_count,
    "empty_response": empty_response_count,
})

{'assembled_issues': 116, 'empty_issue_text': 0, 'empty_response': 0}


## 07. Body Anchor Split

`<조치사항>`과 `<향후 추진계획>` anchor를 기준으로 오른쪽 cell 전체 response를 분리한다. 결합형 `<조치 및 향후계획>`은 분리 불가능성을 숨기지 않고 action 영역으로 보존하며 future는 literal `null`로 둔다.

In [8]:
def split_response(response: str) -> dict[str, str]:
    text = clean_response_text(response)
    action_match = ACTION_ANCHOR_PATTERN.search(text)
    future_match = FUTURE_ANCHOR_PATTERN.search(text)
    combined_match = COMBINED_ANCHOR_PATTERN.search(text)

    action_text = "null"
    future_text = "null"
    anchor_type = "none"

    if action_match:
        action_start = action_match.end()
        action_end = future_match.start() if future_match and future_match.start() > action_start else len(text)
        action_text = normalize_text(text[action_start:action_end]) or "null"
        anchor_type = "action"
    if future_match:
        future_text = normalize_text(text[future_match.end():]) or "null"
        anchor_type = "action_and_future" if action_match else "future_only"
    if not action_match and not future_match and combined_match:
        action_text = normalize_text(text[combined_match.end():]) or "null"
        anchor_type = "combined"

    return {
        "action_text": action_text,
        "future_plan_text": future_text,
        "anchor_type": anchor_type,
    }


anchor_rows = issue_assembly_df["response_native"].map(split_response).apply(pd.Series)
issue_work_df = pd.concat([issue_assembly_df, anchor_rows], axis=1)
anchor_counts = issue_work_df["anchor_type"].value_counts().to_dict()
print({"anchor_counts": anchor_counts})

{'anchor_counts': {'action_and_future': 59, 'action': 54, 'none': 2, 'future_only': 1}}


## 08. Native Status Classification

명시적 미조치/완료 표지를 최우선으로 하고, 그 다음 future/진행 표현, 마지막으로 action의 완료 시제 표현을 적용한다. 어떤 근거에도 맞지 않으면 literal `null`이다.

In [9]:
UNCOMPLETE_STATUS_PATTERN = re.compile(
    r"(?:미\s*조치|미\s*완료|미\s*추진|미\s*실시|조치\s*불가|추진\s*불가|불가능|처리\s*불가)"
)
EXPLICIT_COMPLETE_PATTERN = re.compile(
    r"(?:조치\s*완료|수립\s*완료|시행\s*완료|개정\s*완료|확보\s*완료|완료\s*(?:함|됨|하였|했|되었)?)"
)
ACTIVE_STATUS_PATTERN = re.compile(
    r"(?:조치\s*중|진행\s*중|추진\s*중|검토\s*중|협의\s*중|계속|예정|계획|추진할|실시할|노력할|지속\s*추진)"
)
PAST_ACTION_PATTERN = re.compile(
    r"(?:실시|마련|수립|개정|확보|운영|배포|시행|지원|구축|개최|도입|반영|점검|조사|요청)"
    r"(?:\s*완료|\s*함|\s*됨|\s*하였|\s*했|\s*되었|\s*(?:ㅇ\s*)?\([’'‘\d])"
)


def classify_status(action_text: str, future_plan_text: str, response: str) -> tuple[str, str]:
    combined = normalize_text(response)
    action_normalized = normalize_text(action_text)
    if UNCOMPLETE_STATUS_PATTERN.search(combined):
        return "uncomplete", "explicit_uncomplete"
    if re.match(r"^조치\s*중", action_normalized):
        return "active", "explicit_action_active"
    if re.match(r"^조치\s*완료", action_normalized):
        return "complete", "explicit_action_complete"
    if EXPLICIT_COMPLETE_PATTERN.search(combined):
        return "complete", "explicit_complete"
    if future_plan_text != "null" or ACTIVE_STATUS_PATTERN.search(combined):
        return "active", "active_or_future"
    if action_text != "null" and PAST_ACTION_PATTERN.search(action_text):
        return "complete", "past_action"
    return "null", "no_explicit_evidence"


native_status = issue_work_df.apply(
    lambda r: classify_status(r["action_text"], r["future_plan_text"], r["response_native"]),
    axis=1,
)
issue_work_df["status"] = [x[0] for x in native_status]
issue_work_df["status_rule"] = [x[1] for x in native_status]
issue_work_df["parse_source"] = "native"

print({
    "native_status_counts": issue_work_df["status"].value_counts().reindex(ALLOWED_STATUS, fill_value=0).to_dict(),
    "native_null_status": int(issue_work_df["status"].eq("null").sum()),
})

{'native_status_counts': {'complete': 77, 'active': 39, 'uncomplete': 0, 'null': 0}, 'native_null_status': 0}


## 09. Conditional OCR Fallback

full-page OCR은 실행하지 않는다. native issue/response가 비었거나 anchor와 status가 모두 식별되지 않은 선택 행에 한해 해당 cell rectangle만 Tesseract(`kor+eng`)로 OCR한다. OCR은 native 결과보다 우선하지 않으며, 실제로 회복된 필드만 반영한다.

In [10]:
def ocr_fragments(
    doc: fitz.Document, fragments: list[dict[str, Any]], side: str
) -> tuple[str, str | None]:
    tesseract = shutil.which("tesseract")
    if not tesseract:
        return "", "tesseract executable not found"
    outputs: list[str] = []
    try:
        with tempfile.TemporaryDirectory(prefix="marked_audit_ocr_", dir="/tmp") as temp_dir:
            for i, fragment in enumerate(fragments):
                page = doc[int(fragment["physical_page"]) - 1]
                divider = float(fragment["divider"])
                if side == "left":
                    clip = fitz.Rect(40, float(fragment["top"]), divider, float(fragment["bottom"]))
                else:
                    clip = fitz.Rect(divider, float(fragment["top"]), page.rect.width - 35, float(fragment["bottom"]))
                if clip.height < 3 or clip.width < 3:
                    continue
                png = Path(temp_dir) / f"cell_{i:03d}.png"
                page.get_pixmap(matrix=fitz.Matrix(3, 3), clip=clip, alpha=False).save(png)
                proc = subprocess.run(
                    [tesseract, str(png), "stdout", "-l", "kor+eng", "--psm", "6"],
                    check=False, capture_output=True, text=True, timeout=90,
                )
                if proc.returncode == 0:
                    outputs.append(proc.stdout)
                else:
                    return normalize_text(" ".join(outputs)), normalize_text(proc.stderr)[:500]
    except Exception as exc:
        return normalize_text(" ".join(outputs)), f"{type(exc).__name__}: {exc}"
    return normalize_text(" ".join(outputs)), None


ocr_attempted = 0
ocr_recovered = 0
ocr_failures: list[dict[str, Any]] = []

if RUN_OCR_FALLBACK and not RUN_FULL_PAGE_OCR:
    for idx, row in issue_work_df.iterrows():
        needs_issue_ocr = row["issue_text"] == ""
        needs_right_ocr = (
            row["response_native"] == ""
            or row["status"] == "null"
            or row["action_text"] == "null"
        )
        if not (needs_issue_ocr or needs_right_ocr):
            continue

        ocr_attempted += 1
        row_recovered = False
        if needs_issue_ocr:
            ocr_left, error = ocr_fragments(doc, row["fragments"], "left")
            ocr_left = clean_issue_text(ocr_left, int(row["issue_no"]))
            if ocr_left:
                issue_work_df.at[idx, "issue_text"] = ocr_left
                issue_work_df.at[idx, "parse_source"] = "ocr"
                row_recovered = True
            if error:
                ocr_failures.append({"issue_no": int(row["issue_no"]), "side": "left", "error": error})

        if needs_right_ocr:
            ocr_right, error = ocr_fragments(doc, row["fragments"], "right")
            if ocr_right:
                ocr_split = split_response(ocr_right)
                native_response_empty = row["response_native"] == ""
                if native_response_empty:
                    issue_work_df.at[idx, "response_native"] = ocr_right
                    issue_work_df.at[idx, "action_text"] = ocr_split["action_text"]
                    issue_work_df.at[idx, "future_plan_text"] = ocr_split["future_plan_text"]
                    issue_work_df.at[idx, "anchor_type"] = ocr_split["anchor_type"]
                    issue_work_df.at[idx, "parse_source"] = "ocr"
                    row_recovered = True
                current_status = str(issue_work_df.at[idx, "status"])
                ocr_status, ocr_rule = classify_status(
                    ocr_split["action_text"], ocr_split["future_plan_text"], ocr_right
                )
                if current_status == "null" and ocr_status != "null":
                    issue_work_df.at[idx, "status"] = ocr_status
                    issue_work_df.at[idx, "status_rule"] = f"ocr:{ocr_rule}"
                    if issue_work_df.at[idx, "parse_source"] == "native":
                        issue_work_df.at[idx, "parse_source"] = "native_with_ocr_status"
                    row_recovered = True
            if error:
                ocr_failures.append({"issue_no": int(row["issue_no"]), "side": "right", "error": error})

        if row_recovered:
            ocr_recovered += 1

logger.info("ocr attempted=%d recovered=%d failures=%d", ocr_attempted, ocr_recovered, len(ocr_failures))
print({
    "ocr_attempted": ocr_attempted,
    "ocr_recovered": ocr_recovered,
    "ocr_failures": len(ocr_failures),
})

{'ocr_attempted': 3, 'ocr_recovered': 0, 'ocr_failures': 0}


## 10. Final DataFrame 생성

마킹된 요구사항 한 건을 grain으로 정확히 9개 canonical 컬럼을 생성한다. `issue_no`는 nullable integer, `status`는 category이며 Parquet에는 index를 저장하지 않는다.

In [11]:
canonical_rows = []
for row in issue_work_df.to_dict("records"):
    issue_no = int(row["issue_no"])
    canonical_rows.append({
        "issue_id": f"{meeting_id}_ISSUE_{issue_no:04d}",
        "meeting_id": meeting_id,
        "issue_no": issue_no,
        "issue_text": normalize_text(row["issue_text"]),
        "action_text": row["action_text"] if row["action_text"] else "null",
        "future_plan_text": row["future_plan_text"] if row["future_plan_text"] else "null",
        "status": row["status"] if row["status"] else "null",
        "source_page": int(row["source_page"]),
        "parse_source": row["parse_source"],
    })

marked_issue_df = pd.DataFrame(canonical_rows, columns=CANONICAL_COLUMNS)
marked_issue_df["issue_id"] = marked_issue_df["issue_id"].astype("string")
marked_issue_df["meeting_id"] = marked_issue_df["meeting_id"].astype("string")
marked_issue_df["issue_no"] = marked_issue_df["issue_no"].astype("Int64")
marked_issue_df["source_page"] = marked_issue_df["source_page"].astype("Int64")
marked_issue_df["status"] = pd.Categorical(marked_issue_df["status"], categories=ALLOWED_STATUS)

status_counts = (
    marked_issue_df["status"].value_counts().reindex(ALLOWED_STATUS, fill_value=0).astype(int).to_dict()
)
parse_source_counts = marked_issue_df["parse_source"].value_counts().to_dict()
print({
    "canonical_rows": len(marked_issue_df),
    "status_counts": status_counts,
    "parse_source_counts": parse_source_counts,
})

{'canonical_rows': 116, 'status_counts': {'complete': 77, 'active': 39, 'uncomplete': 0, 'null': 0}, 'parse_source_counts': {'native': 116}}


## 11. Quality Audit

schema, PK, marker coverage, issue text, status, source page와 허용 enum을 검증한다. 원문 번호 결함은 선택 대상 여부와 분리해 기록한다.

In [12]:
quality_checks = {
    "canonical_columns_exact": list(marked_issue_df.columns) == CANONICAL_COLUMNS,
    "review_columns_exact": True,
    "issue_id_unique": bool(marked_issue_df["issue_id"].is_unique),
    "issue_no_unique": bool(marked_issue_df["issue_no"].is_unique),
    "all_selected_issues_mapped": unresolved_issue_count == 0,
    "all_toc_annotations_mapped": len(unresolved_toc_annotations) == 0,
    "all_body_annotations_mapped": len(unresolved_body_annotations) == 0,
    "all_issue_text_present": bool(marked_issue_df["issue_text"].ne("").all()),
    "status_allowed": bool(marked_issue_df["status"].astype("string").isin(ALLOWED_STATUS).all()),
    "parse_source_allowed": bool(marked_issue_df["parse_source"].isin(ALLOWED_PARSE_SOURCE).all()),
    "source_page_in_range": bool(marked_issue_df["source_page"].between(1, PAGE_COUNT).all()),
}
quality_null_counts = {
    "issue_text_empty": int(marked_issue_df["issue_text"].eq("").sum()),
    "action_text_null_literal": int(marked_issue_df["action_text"].eq("null").sum()),
    "future_plan_text_null_literal": int(marked_issue_df["future_plan_text"].eq("null").sum()),
    "status_null_literal": int(marked_issue_df["status"].astype("string").eq("null").sum()),
}
hard_failure = (
    len(annotation_df) == 0
    or len(selected_issue_nos) == 0
    or not quality_checks["issue_id_unique"]
)
partial_condition = (
    unresolved_issue_count > 0
    or quality_null_counts["status_null_literal"] > 0
    or quality_null_counts["issue_text_empty"] > 0
    or DETECTED_REPORT_YEAR != REQUESTED_REPORT_YEAR
)
final_status = "FAILED" if hard_failure else ("PARTIAL" if partial_condition else "COMPLETE")

sample_review = marked_issue_df.sort_values("issue_no").head(10).copy()
sample_review.insert(1, "meeting_year", meeting_year)
sample_review = sample_review[REVIEW_COLUMNS]

logger.info(
    "quality status=%s checks=%s nulls=%s status_counts=%s",
    final_status, quality_checks, quality_null_counts, status_counts,
)
print({
    "quality_checks_passed": sum(bool(v) for v in quality_checks.values()),
    "quality_checks_total": len(quality_checks),
    "null_counts": quality_null_counts,
    "final_status_pre_export": final_status,
})
display(sample_review)

{'quality_checks_passed': 11, 'quality_checks_total': 11, 'null_counts': {'issue_text_empty': 0, 'action_text_null_literal': 3, 'future_plan_text_null_literal': 56, 'status_null_literal': 0}, 'final_status_pre_export': 'COMPLETE'}


,meeting_id,meeting_year,issue_no,issue_text,action_text,future_plan_text,status,source_page
0,UNMAPPED_REPORT_2022,22,5,한국문화관광연구원단기현안 과제수행관련규정명확화필요,조치완료 연구사업추진지침개정(‘23년1월) ㅇ - 지침내단기현안연구(현안과제) 유형...,ㅇ지침준수여부점검등관리·감독철저,complete,27
1,UNMAPPED_REPORT_2022,22,13,소관위원회의인적구성다양화 및운영내실화필요,조치완료 ㅇ문체부소관위원회장애인위원확대계획수립및보고(‘22.11.) - 향후신규위원...,ㅇ위원회별장애인위원위촉확대및현황파악지속추진,complete,28
2,UNMAPPED_REPORT_2022,22,24,세종학당사업수행방식을보조금 에서출연금으로전환하는등 법적근거마련을위한「국어 기본법」개...,조치중 관련국어기본법개정안국회발의 ㅇ - 임오경의원대표발의(’22.11.22) · ...,관련법개정에대해사업운영의효율성및예산운용의 ㅇ 투명성을종합적으로검토해신중하게추진,active,31
3,UNMAPPED_REPORT_2022,22,28,국립세계문자박물관에서현재 까지수집한소장품및관련 용역에대한전수조사실시 하여보고할것,"조치완료 ㅇ소장품수집관련전수조사계획수립(’22.11.2.) - 소장품전수조사, 전시...",null,complete,32
4,UNMAPPED_REPORT_2022,22,52,"BF(배리어프리) 인증실태를 제대로파악하고, 박물관·미술관의 배리어프리확산을위한방안...","조치완료 국·공립박물관및미술관장애인편의시설실태전수조사 ㅇ 및개선방향연구, 현황파악및...",null,complete,38
5,UNMAPPED_REPORT_2022,22,59,"한국콘텐츠진흥원 허위자료 제출, 인사, 급여등문제가 있어철저한조사를통해적법한 조치를취할것","조치중 요청자료(`12년∼`22년승진자명단/등급/연차등)의오류를확인, ㅇ 엄중사과하...","인사관리시스템의승진자동순위산출, 통계화기능등 ㅇ 고도화지속추진(`22.10월∼) -...",active,40
6,UNMAPPED_REPORT_2022,22,64,"영화발전기금고갈위기상황으로, 글로벌OTT 사업자도국내영화 산업발전차원에서기금부과필요","조치중 영화발전기금재원확대를위한사전검토등추진 ㅇ ‘영화’의법적정의재검토*, 온라인영...","온라인영화유통거래정보확보등인프라선행구축및운영 ㅇ 안정화지속 - 다만, OTT 부과금...",active,42
7,UNMAPPED_REPORT_2022,22,74,네이버제트 등 메타버스를 선도하는기업등이자체등급 분류사업자가되는방안등에 대해콘텐츠를...,조치완료 ㅇ‘메타버스와게임물구분등을위한가이드라인’ 마련추진(‘22.9∼) - 관계부...,가이드라인마련을위해현재과기부등관계부처와지속협의 ㅇ,complete,44
8,UNMAPPED_REPORT_2022,22,81,2020년선정된인천시음악 창작소의정상운영에대한 우려및당시공모선정과 정의적정성에대해조사할것,조치완료 ㅇ인천음창소공모선정과정의적정성조사완료(‘22.12월) * ‘20년지역기반형...,캠프마켓마스터플랜연구용역(~’24.2월) 및인천시음악 ㅇ 창작소건물보존검토중,complete,46
9,UNMAPPED_REPORT_2022,22,82,"온라인암표신고게시판의 실효성이없어, 이를보완할 온라인암표매매실태및 피해규모파악, 온...",조치중 한국콘텐츠진흥원과협의하여매크로의심사례를실효성 ㅇ 있게선별할수있도록온라인암표신...,ㅇ향후티켓예매처· 경찰청등관계기관협의를통해실효성 있는암표근절대책마련예정,active,47


## 11A. Marked Issue 2차 전수검토

선택된 모든 issue를 annotation 증거, exact 번호 매핑, issue/response 존재, status enum, source page 범위 기준으로 다시 검토한다. 이 검토는 최종 8컬럼 CSV를 변경하지 않고 manifest와 품질 보고서에 감사 결과를 남긴다.

In [13]:
second_pass_rows = []
for row in issue_work_df.to_dict("records"):
    issue_no = int(row["issue_no"])
    mapping_row = mapping_df.loc[mapping_df["issue_no"].eq(issue_no)].iloc[0]
    annotation_evidence_count = int(
        toc_df.loc[toc_df["issue_no"].eq(issue_no), "marker_count"].sum()
    ) + int(body_marker_counts.get(issue_no, 0))
    action_normalized = normalize_text(row["action_text"])
    explicit_status_consistent = not (
        (re.match(r"^조치\s*중", action_normalized) and row["status"] != "active")
        or (re.match(r"^조치\s*완료", action_normalized) and row["status"] != "complete")
    )
    checks = {
        "annotation_evidence": annotation_evidence_count > 0,
        "exact_issue_mapping": mapping_row["mapping_method"] == "issue_no_exact",
        "issue_text_present": normalize_text(row["issue_text"]) != "",
        "response_present": normalize_text(row["response_native"]) != "",
        "status_allowed": str(row["status"]) in ALLOWED_STATUS,
        "explicit_status_consistent": explicit_status_consistent,
        "source_page_valid": 1 <= int(row["source_page"]) <= PAGE_COUNT,
    }
    second_pass_rows.append({
        "issue_no": issue_no,
        "marker_source": mapping_row["marker_source"],
        "annotation_evidence_count": annotation_evidence_count,
        **checks,
        "review_pass": all(checks.values()),
    })

second_pass_review_df = pd.DataFrame(second_pass_rows).sort_values("issue_no").reset_index(drop=True)
second_pass_flagged = second_pass_review_df.loc[
    ~second_pass_review_df["review_pass"], "issue_no"
].astype(int).tolist()
second_pass_reviewed_count = len(second_pass_review_df)
second_pass_pass_count = int(second_pass_review_df["review_pass"].sum())

quality_checks["second_pass_all_selected_reviewed"] = (
    second_pass_reviewed_count == len(selected_issue_nos)
)
quality_checks["second_pass_all_passed"] = len(second_pass_flagged) == 0
if second_pass_flagged and final_status != "FAILED":
    final_status = "PARTIAL"

logger.info(
    "second_pass reviewed=%d passed=%d flagged=%s",
    second_pass_reviewed_count, second_pass_pass_count, second_pass_flagged,
)
print({
    "second_pass_reviewed": second_pass_reviewed_count,
    "second_pass_passed": second_pass_pass_count,
    "second_pass_flagged": second_pass_flagged,
})

{'second_pass_reviewed': 116, 'second_pass_passed': 116, 'second_pass_flagged': []}


## 12. Export·Manifest·Run Report

canonical Parquet, BOM 포함 8컬럼 CSV, 품질 보고서, manifest, 실행 보고서를 저장한 뒤 CSV/Parquet를 다시 읽어 schema·row count·PK를 검증한다.

In [14]:
def markdown_table(df: pd.DataFrame) -> str:
    if df.empty:
        return "_no rows_"
    safe = df.astype("string").fillna("").apply(
        lambda col: col.str.replace("|", "\\|", regex=False).str.replace("\n", " ", regex=False)
    )
    headers = list(safe.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    lines.extend("| " + " | ".join(map(str, row)) + " |" for row in safe.itertuples(index=False, name=None))
    return "\n".join(lines)


def output_sha(path: Path) -> str | None:
    return sha256_file(path) if path.exists() and path.is_file() else None


if not RUN_EXPORT:
    raise RuntimeError("RUN_EXPORT=False is incompatible with this full-cycle request")

marked_issue_df.to_parquet(PARQUET_PATH, index=False)

review_df = marked_issue_df.copy()
review_df.insert(1, "meeting_year", meeting_year)
review_df = review_df[REVIEW_COLUMNS].copy()
for col in ["action_text", "future_plan_text", "status"]:
    review_df[col] = review_df[col].astype("string").fillna("null").replace("", "null")
review_df.to_csv(REVIEW_CSV_PATH, encoding="utf-8-sig", index=False)

reloaded_parquet = pd.read_parquet(PARQUET_PATH)
reloaded_csv = pd.read_csv(
    REVIEW_CSV_PATH,
    encoding="utf-8-sig",
    dtype={"meeting_id": "string", "meeting_year": "string", "issue_no": "Int64", "source_page": "Int64"},
    keep_default_na=False,
)
reload_checks = {
    "parquet_exists": PARQUET_PATH.exists(),
    "csv_exists": REVIEW_CSV_PATH.exists(),
    "parquet_rows_match": len(reloaded_parquet) == len(marked_issue_df),
    "csv_rows_match": len(reloaded_csv) == len(marked_issue_df),
    "parquet_columns_exact": list(reloaded_parquet.columns) == CANONICAL_COLUMNS,
    "csv_columns_exact": list(reloaded_csv.columns) == REVIEW_COLUMNS,
    "parquet_issue_id_unique": bool(reloaded_parquet["issue_id"].is_unique),
    "csv_literal_null_contract": bool(
        reloaded_csv[["action_text", "future_plan_text", "status"]]
        .apply(lambda s: ~s.isin(["", "NaN", "None", "<NA>"]))
        .all().all()
    ),
}
if not all(reload_checks.values()):
    final_status = "FAILED"

quality_report = f"""# Marked Mapping Quality

## 1. Input
- Canonical PDF: `{input_pdf}`
- SHA-256: `{PDF_SHA256}`
- Pages: {PAGE_COUNT}
- Requested year: {REQUESTED_REPORT_YEAR}
- Detected title-page year: {DETECTED_REPORT_YEAR}
- Sequential PDF audits: {len(pdf_audits)}

## 2. Annotation and row mapping
- Highlight annotations: {len(annotation_df)}
- Front-matter non-selector annotations: {front_matter_annotation_count}
- TOC annotation mapping: {len(toc_annotation_map)} mapped / {len(unresolved_toc_annotations)} unresolved
- Body annotation mapping: {len(body_annotation_map)} mapped / {len(unresolved_body_annotations)} unresolved
- Selected issues: {len(selected_issue_nos)}
- Issue mapping: {mapped_issue_count} mapped / {unresolved_issue_count} unresolved
- Marker sources: {json.dumps(marker_source_counts, ensure_ascii=False)}
- Native numbering audit: {json.dumps(body_numbering_defects, ensure_ascii=False)}

## 3. Parsing quality
- Native rows: {int((marked_issue_df["parse_source"] == "native").sum())}
- Second-pass reviewed: {second_pass_reviewed_count}
- Second-pass passed: {second_pass_pass_count}
- Second-pass flagged: {second_pass_flagged}
- OCR attempted: {ocr_attempted}
- OCR recovered: {ocr_recovered}
- OCR failures: {len(ocr_failures)}
- Null counts: {json.dumps(quality_null_counts, ensure_ascii=False)}

## 4. Status
- complete: {status_counts.get("complete", 0)}
- active: {status_counts.get("active", 0)}
- uncomplete: {status_counts.get("uncomplete", 0)}
- null: {status_counts.get("null", 0)}

## 5. Checks
{chr(10).join(f"- {k}: {'PASS' if v else 'FAIL'}" for k, v in {**quality_checks, **reload_checks}.items())}

## 6. Warnings
{chr(10).join(f"- {w}" for w in warnings) if warnings else "- None"}

## 7. Final status
- **{final_status}**
"""
QUALITY_REPORT_PATH.write_text(quality_report, encoding="utf-8")

manifest = {
    "pipeline": f"marked_audit_result_index_{REQUESTED_REPORT_YEAR}",
    "requested_report_year": REQUESTED_REPORT_YEAR,
    "detected_report_year": DETECTED_REPORT_YEAR,
    "run_started_at": RUN_STARTED_AT.isoformat(),
    "run_finished_at": datetime.now().astimezone().isoformat(),
    "project_root": str(PROJECT_ROOT),
    "notebook": str(NOTEBOOK_PATH),
    "input_selection": {
        "source_directory": str(pdf_source_dir),
        "selection_rule": pdf_selection_rule,
        "canonical_pdf": str(input_pdf),
        "sha256": PDF_SHA256,
        "page_count": PAGE_COUNT,
        "sequential_pdf_audits": pdf_audits,
    },
    "registry": {
        "path": str(registry_path) if registry_path else None,
        "mapping_status": registry_mapping_status,
        "mapping_warning": registry_mapping_warning,
        "meeting_id": meeting_id,
        "meeting_year": meeting_year,
        "matched_candidate_count": len(registry_candidates_matched),
    },
    "flags": {k: v for k, v in globals().items() if k.startswith("RUN_") and isinstance(v, bool)},
    "counts": {
        "highlight_annotations": len(annotation_df),
        "front_matter_annotations": front_matter_annotation_count,
        "toc_annotations": len(toc_annotation_map),
        "body_annotations": len(body_annotation_map),
        "selected_issues": len(selected_issue_nos),
        "mapped_issues": mapped_issue_count,
        "unresolved_issues": unresolved_issue_count,
        "native_parsed": int((marked_issue_df["parse_source"] == "native").sum()),
        "ocr_attempted": ocr_attempted,
        "ocr_recovered": ocr_recovered,
        "status": status_counts,
        "nulls": quality_null_counts,
        "second_pass_reviewed": second_pass_reviewed_count,
        "second_pass_passed": second_pass_pass_count,
        "second_pass_flagged": second_pass_flagged,
    },
    "mapping": {
        "marker_source_counts": marker_source_counts,
        "front_matter_non_selector_annotations": front_matter_annotation_count,
        "unresolved_toc_annotations": len(unresolved_toc_annotations),
        "unresolved_body_annotations": len(unresolved_body_annotations),
        "body_numbering_defects": body_numbering_defects,
    },
    "schemas": {
        "canonical_columns": CANONICAL_COLUMNS,
        "review_columns": REVIEW_COLUMNS,
    },
    "quality_checks": quality_checks,
    "reload_checks": reload_checks,
    "ocr_failures": ocr_failures,
    "warnings": warnings,
    "outputs": {
        "parquet": str(PARQUET_PATH),
        "review_csv": str(REVIEW_CSV_PATH),
        "quality_report": str(QUALITY_REPORT_PATH),
        "run_report": str(RUN_REPORT_PATH),
        "log": str(LOG_PATH),
        "sha256": {
            "parquet": output_sha(PARQUET_PATH),
            "review_csv": output_sha(REVIEW_CSV_PATH),
            "quality_report": output_sha(QUALITY_REPORT_PATH),
        },
    },
    "final_status": final_status,
}
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

run_report = f"""# Marked Audit Result Index Run Report

## 1. 실행 개요
- PROJECT_ROOT: `{PROJECT_ROOT}`
- Notebook: `{NOTEBOOK_PATH}`
- Canonical PDF: `{input_pdf}`
- PDF SHA-256: `{PDF_SHA256}`
- Page count: {PAGE_COUNT}
- Registry: `{registry_path if registry_path else 'not found'}`
- meeting_id: `{meeting_id}`
- Registry mapping: {registry_mapping_status}

## 2. PDF/Annotation
- 공급 PDF 감사: {len(pdf_audits)}건
- Highlight annotation: {len(annotation_df)}
- 표지 non-selector annotation: {front_matter_annotation_count}
- 목차 annotation: {len(toc_annotation_map)}
- 본문 annotation: {len(body_annotation_map)}
- annotation unresolved: {len(unresolved_toc_annotations) + len(unresolved_body_annotations)}

## 3. Mapping 및 OCR
- 선택 issue: {len(selected_issue_nos)}
- Mapping 성공: {mapped_issue_count}
- Mapping unresolved: {unresolved_issue_count}
- Native parsed: {int((marked_issue_df["parse_source"] == "native").sum())}
- OCR 시도: {ocr_attempted}
- OCR 성공: {ocr_recovered}
- OCR failure: {len(ocr_failures)}
- 2차 전수검토: {second_pass_pass_count}/{second_pass_reviewed_count} PASS
- 2차 검토 flagged: {second_pass_flagged}

## 4. 상태 분류
- complete: {status_counts.get("complete", 0)}
- active: {status_counts.get("active", 0)}
- uncomplete: {status_counts.get("uncomplete", 0)}
- null: {status_counts.get("null", 0)}

## 5. 생성 파일
- Parquet: `{PARQUET_PATH}`
- CSV: `{REVIEW_CSV_PATH}`
- Manifest: `{MANIFEST_PATH}`
- Quality Report: `{QUALITY_REPORT_PATH}`
- Notebook: `{NOTEBOOK_PATH}`
- Log: `{LOG_PATH}`

## 6. 최종 샘플
{markdown_table(reloaded_csv.sort_values("issue_no").head(10)[REVIEW_COLUMNS])}

## 7. 경고
{chr(10).join(f"- {w}" for w in warnings) if warnings else "- None"}

## 8. 최종 상태
- **{final_status}**
"""
RUN_REPORT_PATH.write_text(run_report, encoding="utf-8")

# Report와 manifest까지 생성된 최종 파일 존재성을 다시 확인한다.
final_file_checks = {
    "parquet": PARQUET_PATH.exists(),
    "review_csv": REVIEW_CSV_PATH.exists(),
    "manifest": MANIFEST_PATH.exists(),
    "quality_report": QUALITY_REPORT_PATH.exists(),
    "run_report": RUN_REPORT_PATH.exists(),
    "log": LOG_PATH.exists(),
}
if not all(final_file_checks.values()):
    raise RuntimeError(f"Output file creation failed: {final_file_checks}")

logger.info("pipeline_end status=%s files=%s", final_status, final_file_checks)
for handler in logger.handlers:
    handler.flush()

print({
    "final_status": final_status,
    "rows": len(marked_issue_df),
    "reload_checks": reload_checks,
    "files": {k: str(v) for k, v in {
        "parquet": PARQUET_PATH,
        "review_csv": REVIEW_CSV_PATH,
        "manifest": MANIFEST_PATH,
        "quality_report": QUALITY_REPORT_PATH,
        "run_report": RUN_REPORT_PATH,
        "log": LOG_PATH,
    }.items()},
    "review_columns": REVIEW_COLUMNS,
})

{'final_status': 'COMPLETE', 'rows': 116, 'reload_checks': {'parquet_exists': True, 'csv_exists': True, 'parquet_rows_match': True, 'csv_rows_match': True, 'parquet_columns_exact': True, 'csv_columns_exact': True, 'parquet_issue_id_unique': True, 'csv_literal_null_contract': True}, 'files': {'parquet': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/outputs/marked_parallel/2022/2022_marked_issue_mapping.parquet', 'review_csv': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/outputs/marked_parallel/2022/2022_marked_issue_mapping.csv', 'manifest': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/outputs/marked_parallel/2022/2022_pipeline_manifest.json', 'quality_report': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/reports/2022_MARKED_MAPPING_QUALITY.md', 'run_report': '/home/sieg/projects-wsl/SBS_dataScience/DSJA/P3_CULTURE/P3_TARGET/outputs/marked_parallel/2022/2022_RUN_REPORT.md', 'log': '/home/sieg/projects